# Autoencoder — MNIST ile Boyut İndirgeme ve Gürültü Giderme\nEncoder → Latent → Decoder yapısı. Girişi sıkıştırıp yeniden oluşturmayı öğrenir.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim\nimport torchvision, torchvision.transforms as T\nimport matplotlib.pyplot as plt, numpy as np, os\nos.makedirs('cikti', exist_ok=True)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Cihaz: {device}')

## MNIST Veri Seti

In [ ]:
transform = T.Compose([T.ToTensor()])\ntrain_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)\ntest_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)\ntrain_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)\ntest_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False)\nprint(f'Eğitim: {len(train_set)}, Test: {len(test_set)}')

In [ ]:
imgs, _ = next(iter(train_loader))\nfig, axes = plt.subplots(2, 5, figsize=(10, 4))\nfor i, ax in enumerate(axes.flat):\n    ax.imshow(imgs[i].squeeze(), cmap='gray'); ax.axis('off')\nplt.suptitle('MNIST Örnekleri')\nplt.show()

## Autoencoder Modeli

In [ ]:
class Autoencoder(nn.Module):\n    def __init__(self):\n        super().__init__()\n        # Encoder: 784 -> 128 -> 64 -> 32\n        self.encoder = nn.Sequential(\n            nn.Linear(784, 128), nn.ReLU(),\n            nn.Linear(128, 64), nn.ReLU(),\n            nn.Linear(64, 32),  # latent boyut = 32\n        )\n        # Decoder: 32 -> 64 -> 128 -> 784\n        self.decoder = nn.Sequential(\n            nn.Linear(32, 64), nn.ReLU(),\n            nn.Linear(64, 128), nn.ReLU(),\n            nn.Linear(128, 784), nn.Sigmoid(),\n        )\n    def forward(self, x):\n        x = x.view(-1, 784)\n        encoded = self.encoder(x)\n        decoded = self.decoder(encoded)\n        return decoded.view(-1, 1, 28, 28)\n\nmodel = Autoencoder().to(device)\nprint(f'Parametre: {sum(p.numel() for p in model.parameters()):,}')

## Eğitim

In [ ]:
criterion = nn.MSELoss()\noptimizer = optim.Adam(model.parameters(), lr=0.001)\nlosses = []\nEPOCHS = 10\n\nfor epoch in range(EPOCHS):\n    model.train()\n    epoch_loss = 0\n    for imgs, _ in train_loader:\n        imgs = imgs.to(device)\n        optimizer.zero_grad()\n        out = model(imgs)\n        loss = criterion(out, imgs)\n        loss.backward()\n        optimizer.step()\n        epoch_loss += loss.item()\n    losses.append(epoch_loss / len(train_loader))\n    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {losses[-1]:.6f}')\n\nplt.plot(losses); plt.title('Eğitim Loss'); plt.xlabel('Epoch'); plt.show()

## Orijinal vs Yeniden Oluşturulan

In [ ]:
test_imgs, _ = next(iter(test_loader))\ntest_imgs = test_imgs[:10].to(device)\nmodel.eval()\nwith torch.no_grad():\n    reconstructed = model(test_imgs)\n\nfig, axes = plt.subplots(2, 10, figsize=(15, 3))\nfor i in range(10):\n    axes[0, i].imshow(test_imgs[i].cpu().squeeze(), cmap='gray')\n    axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')\n    axes[0, i].axis('off'); axes[1, i].axis('off')\naxes[0, 0].set_title('Orijinal', y=1.1, fontsize=10)\naxes[1, 0].set_title('Rekonstrükte', y=1.1, fontsize=10)\nplt.savefig('cikti/autoencoder.png', dpi=100, bbox_inches='tight')\nplt.show()

## Denoising Autoencoder (Gürültü Giderme)

In [ ]:
# Test görüntülerine gürültü ekle ve temizle\nnoise_factor = 0.4\nnoisy = test_imgs + noise_factor * torch.randn_like(test_imgs)\nnoisy = torch.clamp(noisy, 0., 1.)\n\nwith torch.no_grad():\n    denoised = model(noisy)\n\nfig, axes = plt.subplots(3, 10, figsize=(15, 5))\nfor i in range(10):\n    axes[0, i].imshow(test_imgs[i].cpu().squeeze(), cmap='gray')\n    axes[1, i].imshow(noisy[i].cpu().squeeze(), cmap='gray')\n    axes[2, i].imshow(denoised[i].cpu().squeeze(), cmap='gray')\n    for j in range(3): axes[j, i].axis('off')\naxes[0, 0].set_title('Orijinal', y=1.1)\naxes[1, 0].set_title('Gürültülü', y=1.1)\naxes[2, 0].set_title('Temizlenmiş', y=1.1)\nplt.savefig('cikti/denoising.png', dpi=100, bbox_inches='tight')\nplt.show()\nprint('Autoencoder eğitimi tamamlandı.')